# Skin Cancer Detection: Kaggle eğitim koşusu

Bu notebook Mac'ten `kaggle kernels push -p kaggle` ile gönderilip Kaggle'da arka planda çalışır
(tarayıcı açık kalmak zorunda değil). Colab notebook'undan farkları:
- Veri Kaggle'daki HAM10000 veri setinden doğrudan bağlanır; API key gerekmez.
- Kod `/tmp`'ye klonlanır, böylece Kaggle çıktısına sadece sonuçlar (`/kaggle/working/models`) girer.
- Sonuçlar Mac'ten `kaggle kernels output yunusemreozkan/skin-cancer-train` ile çekilir.

Hangi modelin eğitileceği aşağıdaki ayar hücresinde.

**v2 / C0 (2026-11-17'den itibaren): 5-fold CV koşusu.** Bu notebook test setini **ölçmez** (eskiden her koşuda
otomatik ölçüyordu; denetim K1). Her fold'un son epoch modeli ve OOF logit'leri `/kaggle/working/cv/fold{k}/` altına yazılır.

In [ ]:
# Bu kosunun ayarlari.
ARCHITECTURE = 'convnext_tiny_in22k_384'  # timm, ImageNet-22k + 384px
# v2 / C0 baseline CV (2026-11-17): final_v1 uyelerinin tarifi (tarif v2), seed 42, EMA yok.
# Her fold: diger 4 fold egitim, bu fold dogrulama (OOF). Son epoch kaydedilir (en iyi epoch secilmez); test tahmin EDILMEZ.
CV_FOLDS_FILE = 'splits/cv5_v1.csv'
FOLDS = [3, 4]  # bu kosuda egitilecek fold'lar. Kosu 1: [0, 1, 2] (Version 11, bitti); kosu 2: [3, 4]
TRAIN_ARGS = ('--feature-extract-epochs 5 --finetune-epochs 25 --finetune-lr 1e-4 --scheduler warmup_cosine '
              '--warmup-epochs 2 --label-smoothing 0.1 --patience 30 --seed 42')
# True: ISIC 2019'un HAM disi goruntuleri (SCC haric) SADECE train'e eklenir (v2 deney 1 icin; baseline'da False).
USE_ISIC2019 = False

In [ ]:
!nvidia-smi

In [ ]:
%cd /tmp
!rm -rf skin-cancer-detection
!git clone --depth 1 https://github.com/yunus345/skin-cancer-detection.git
%cd /tmp/skin-cancer-detection
!git log --oneline -1

In [ ]:
# Kaggle'da torch/torchvision CUDA'li kurulu geliyor; augmentation ve timm'i yerelde test ettigimiz surume sabitliyoruz.
!pip install -q albumentations==2.0.8 timm==1.0.30
# Bu kosunun butun paket surumleri: requirements.txt'yi sabitlemek icin (ROADMAP A4).
!mkdir -p /kaggle/working && pip freeze > /kaggle/working/pip_freeze.txt
!python -c "import torch, timm, albumentations; print(torch.__version__, timm.__version__, albumentations.__version__)"

In [ ]:
import glob, os

# Kaggle'in veri setini bagladigi klasoru metadata dosyasindan buluyoruz (yol zamanla degisebiliyor).
matches = glob.glob('/kaggle/input/**/HAM10000_metadata.csv', recursive=True)
assert matches, 'HAM10000_metadata.csv bulunamadi: veri seti bagli mi?'
DATA_ROOT = os.path.dirname(matches[0])
print('Veri klasoru:', DATA_ROOT)

!python -m src.data.build_manifest --data-root {DATA_ROOT} --out data/processed/manifest.csv
!python -m src.data.split_manifest --manifest data/processed/manifest.csv --out data/processed/manifest.csv
!python -m src.data.validate_manifest --manifest data/processed/manifest.csv

TRAIN_MANIFEST = 'data/processed/manifest.csv'
if USE_ISIC2019:
    # Goruntuler Kaggle'daki kopyadan (andrewmvd/isic-2019); etiket ve metadata ISIC'in resmi adresinden.
    images = glob.glob('/kaggle/input/**/ISIC_2019_Training_Input/ISIC_0000000.jpg', recursive=True)
    assert images, 'ISIC 2019 goruntuleri bulunamadi: veri seti bagli mi?'
    ISIC2019_IMAGES = os.path.dirname(images[0])
    !mkdir -p data/external/isic2019
    !curl -sS -o data/external/isic2019/gt.csv https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv
    !curl -sS -o data/external/isic2019/meta.csv https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Metadata.csv
    !python -m src.data.add_isic2019 --ground-truth data/external/isic2019/gt.csv --metadata data/external/isic2019/meta.csv --images-dir {ISIC2019_IMAGES} --out data/processed/manifest_isic2019.csv
    # Goruntu varligi add_isic2019'da kontrol edildi; burada sizinti ve dis test kontrolu.
    !python -m src.data.validate_manifest --manifest data/processed/manifest_isic2019.csv --skip-image-check
    TRAIN_MANIFEST = 'data/processed/manifest_isic2019.csv'
print('Egitim manifest:', TRAIN_MANIFEST)

In [ ]:
# Kaggle GPU makinelerinde 4 CPU cekirdegi var; veri yuklemeyi 4 yardimci surece dagitiyoruz.
# Her fold dogrudan /kaggle/working altina yazilir: sonraki fold'da oturum cokse bile biten fold'lar kaybolmaz.
for k in FOLDS:
    OUT = f'/kaggle/working/cv/fold{k}'
    print(f'===== fold {k} =====', flush=True)
    !python -m src.training.train --architecture {ARCHITECTURE} --manifest {TRAIN_MANIFEST} --num-workers 4 --device auto --cv-folds {CV_FOLDS_FILE} --fold {k} --output-dir {OUT} {TRAIN_ARGS}

In [ ]:
# Ozet: her fold'un son epoch val (OOF) metrikleri. Test seti burada ve bu notebook'un hicbir yerinde olculmez.
import json, os
for k in FOLDS:
    path = f'/kaggle/working/cv/fold{k}/metrics.json'
    if os.path.exists(path):
        m = json.load(open(path))
        print(f"fold {k}: F1 {m['f1']:.4f}  BA {m['balanced_accuracy']:.4f}  epoch {m['epoch']}")
    else:
        print(f'fold {k}: CIKTI YOK (egitim hata vermis olabilir, yukaridaki logu kontrol et)')

In [ ]:
!ls -la /kaggle/working/cv/*